In [0]:
import pytest
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def test_impossible_speed_detected(spark):
    df = spark.createDataFrame([
        Row(trip_id="T1", average_speed_kmh=45.0),
        Row(trip_id="T2", average_speed_kmh=250.0),
    ])
    SPEED_THRESHOLD = 100
    result = df.withColumn(
        "anomaly_type",
        F.when(F.col("average_speed_kmh") > SPEED_THRESHOLD, "IMPOSSIBLE_SPEED").otherwise(None)
    ).collect()
    assert result[0]["anomaly_type"] is None
    assert result[1]["anomaly_type"] == "IMPOSSIBLE_SPEED"


def test_latest_trip_state_selection(spark):
    df = spark.createDataFrame([
        Row(trip_id="T1", event_type="TRIP_REQUESTED", stage_rank=1),
        Row(trip_id="T1", event_type="DRIVER_ASSIGNED", stage_rank=2),
        Row(trip_id="T1", event_type="TRIP_COMPLETED", stage_rank=4),
    ])
    window_spec = Window.partitionBy("trip_id").orderBy(F.col("stage_rank").desc())
    result = (
        df.withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .collect()
    )
    assert len(result) == 1
    assert result[0]["event_type"] == "TRIP_COMPLETED"

In [0]:
def run_test(test_func, name):
    try:
        test_func(spark)
        print(f"✅ PASS: {name}")
    except AssertionError as e:
        print(f"❌ FAIL: {name} — {e}")
    except Exception as e:
        print(f"⚠️ ERROR: {name} — {e}")

run_test(test_impossible_speed_detected, "test_impossible_speed_detected")
run_test(test_latest_trip_state_selection, "test_latest_trip_state_selection")